<a href="https://colab.research.google.com/github/simplyshree/SeqTrainer/blob/issue-3-all-model-baselines/notebooks/final_training/dnabert2_final_training_t4_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Final DNABERT2 promoter benchmark: T4

This notebook follows the same Colab setup pattern as the working DNABERT2 T4 notebook, but runs the stronger staged fine-tuning profile. It uses the exact CNN-v2 train/validation/test CSVs, seed 42, validation-only MCC/AUPRC selection, and one final held-out test evaluation.

Do not compare a new result with CNN-v2 unless the split audit in `input_split_audit.json` reports the canonical row counts and matching source hashes.


## 1. Verify the Colab T4 accelerator


In [ ]:
import subprocess

gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    text=True,
).strip()
print(gpu_info)
if "T4" not in gpu_info:
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")


## 2. Set paths and the final training profile

The defaults are the controlled final candidate: six maximum epochs, one head-only epoch, then the top four encoder layers. Changing a profile value requires `RESUME_MODE = "none"` or a new output directory; the runner deliberately refuses to mix incompatible checkpoints.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
BRANCH = "issue-3-all-model-baselines"
REPO_DIR = Path("/content/SeqTrainer")
MINIFORGE_DIR = Path("/content/miniforge3")
ENV_DIR = Path("/content/envs/seqtrainer-final-dnabert2-t4")
ENV_PYTHON = ENV_DIR / "bin" / "python"

# Set this only when automatic Drive discovery finds more than one complete split directory.
DRIVE_DATA_DIR = None

RESUME_MODE = "latest"  # latest, best, or none
LOSS_MODE = "bce"       # BCE is the default because the shared training split is balanced.
RUN_MAX_LENGTH_128_CANDIDATE = False

# Controlled final profile. Keep these values unchanged for the first final run.
MAX_EPOCHS = 6
PATIENCE = 2
HEAD_ONLY_EPOCHS = 1
UNFREEZE_TOP_LAYERS = 4
HEAD_LEARNING_RATE = 1e-4
ENCODER_LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
DROPOUT = 0.20
PHYSICAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 16  # effective batch size = 32
WARMUP_RATIO = 0.08

LOCAL_DATA_DIR = Path("/content/seqtrainer_final_training/data")
LOCAL_OUTPUT_DIR = Path("/content/seqtrainer_final_training/dnabert2_seed42")
LOCAL_HF_HOME = Path("/content/seqtrainer_final_training/huggingface")

print("Profile: staged DNABERT2, effective batch", PHYSICAL_BATCH_SIZE * GRADIENT_ACCUMULATION)


## 3. Create the pinned Python 3.10 environment


In [ ]:
installer = Path("/content/Miniforge3-Linux-x86_64.sh")
if not (MINIFORGE_DIR / "bin" / "conda").exists():
    subprocess.run(["wget", "-q", "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh", "-O", str(installer)], check=True)
    subprocess.run(["bash", str(installer), "-b", "-p", str(MINIFORGE_DIR)], check=True)
conda = MINIFORGE_DIR / "bin" / "conda"
if not ENV_PYTHON.exists():
    subprocess.run([str(conda), "create", "-y", "-p", str(ENV_DIR), "python=3.10", "pip"], check=True)

install_env = os.environ.copy()
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools<70", "wheel"], check=True, env=install_env)
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "torch==2.2.2", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True, env=install_env)
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "transformers==4.29.2", "numpy==1.24.4", "pandas==2.0.3", "scikit-learn==1.3.2", "einops==0.6.1", "rdflib", "matplotlib"], check=True, env=install_env)
subprocess.run([str(ENV_PYTHON), "-m", "pip", "uninstall", "-y", "triton", "flash-attn", "flash_attn"], check=False, env=install_env)
print("Pinned environment:", ENV_PYTHON)


## 4. Check out SeqTrainer and verify the isolated environment


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
subprocess.run([str(ENV_PYTHON), "-m", "pip", "install", "-e", str(REPO_DIR), "--no-deps"], check=True, env=install_env)

check = r'''
import json, sys, torch, transformers, seqtrainer
payload = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "seqtrainer": seqtrainer.__file__,
}
print(json.dumps(payload, indent=2))
assert payload["cuda"] and "T4" in payload["gpu"]
'''
subprocess.run([str(ENV_PYTHON), "-c", check], check=True, env=install_env)
print("Commit:", subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip())


## 5. Mount Drive and locate the three shared CSV splits

The notebook searches `AIxBio`, `AI x Bio`, MyDrive, and Shared drives. If exactly one directory has all three required files it selects it. If more than one is found, set `DRIVE_DATA_DIR` in cell 2 to the intended folder and rerun this cell.


In [ ]:
from google.colab import drive
import time

MOUNT_POINT = Path("/content/drive")
already_mounted = os.path.ismount(str(MOUNT_POINT)) or (MOUNT_POINT / "MyDrive").exists()
if not already_mounted:
    if MOUNT_POINT.exists() and not MOUNT_POINT.is_symlink() and any(MOUNT_POINT.iterdir()):
        stale = Path(f"/content/drive_stale_{int(time.time())}")
        MOUNT_POINT.rename(stale)
        print("Moved stale mount directory to:", stale)
    MOUNT_POINT.mkdir(parents=True, exist_ok=True)
    drive.mount(str(MOUNT_POINT), force_remount=False)
else:
    print("Google Drive is already mounted.")

MY_DRIVE = Path("/content/drive/MyDrive")
SHARED_DRIVES = Path("/content/drive/Shareddrives")
DRIVE_OUTPUT_DIR = MY_DRIVE / "SeqTrainer" / "final_training" / "dnabert2_seed42"
DRIVE_MODEL_CACHE = MY_DRIVE / "SeqTrainer" / "final_training" / "model_cache" / "dnabert2"
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_MODEL_CACHE.mkdir(parents=True, exist_ok=True)

SPLIT_FILES = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}

def contains_all_splits(directory):
    directory = Path(directory)
    return directory.is_dir() and all((directory / name).is_file() for name in SPLIT_FILES.values())

def discover_split_directories():
    if DRIVE_DATA_DIR is not None:
        candidate = Path(DRIVE_DATA_DIR)
        if not contains_all_splits(candidate):
            raise FileNotFoundError(f"DRIVE_DATA_DIR does not contain all required splits: {candidate}")
        return [candidate]
    matches = []
    for root in (MY_DRIVE / "AIxBio", MY_DRIVE / "AI x Bio", MY_DRIVE, SHARED_DRIVES):
        if not root.exists():
            continue
        for train_path in root.rglob(SPLIT_FILES["train"]):
            candidate = train_path.parent.resolve()
            if contains_all_splits(candidate) and candidate not in matches:
                matches.append(candidate)
        if matches and root in (MY_DRIVE / "AIxBio", MY_DRIVE / "AI x Bio"):
            break
    return matches

matches = discover_split_directories()
if not matches:
    raise FileNotFoundError("No Drive directory contains all three shared benchmark CSVs.")
if len(matches) > 1:
    print("Multiple complete split directories were found:")
    for path in matches:
        print("-", path)
    raise RuntimeError("Set DRIVE_DATA_DIR in cell 2 to exactly one directory above, then rerun this cell.")

DRIVE_DATA_DIR = matches[0]
LOCAL_HF_HOME.mkdir(parents=True, exist_ok=True)
if any(DRIVE_MODEL_CACHE.iterdir()):
    shutil.copytree(DRIVE_MODEL_CACHE, LOCAL_HF_HOME, dirs_exist_ok=True)
    print("Restored DNABERT2 cache from Drive.")
print("Using shared splits from:", DRIVE_DATA_DIR)
print("Persistent outputs:", DRIVE_OUTPUT_DIR)


## 6. Run staged final training

BCE is the primary candidate because the training labels are nearly balanced. The optional 128-token comparison trains two validation-only candidates, picks the better validation MCC (then AUPRC), and performs one test evaluation for the winner.


In [ ]:
import json

helper = REPO_DIR / "notebooks/final_training/helpers/run_dnabert2_final.py"
run_env = os.environ.copy()
run_env["HF_HOME"] = str(LOCAL_HF_HOME)
run_env["TRANSFORMERS_CACHE"] = str(LOCAL_HF_HOME)
run_env["TOKENIZERS_PARALLELISM"] = "false"
run_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    run_env["HF_TOKEN"] = hf_token
    run_env["HUGGING_FACE_HUB_TOKEN"] = hf_token
    print("HF_TOKEN loaded from Colab Secrets.")
else:
    print("No HF_TOKEN found; the public pinned model download will be used.")

def run_candidate(max_length, drive_output, local_output, selection_only):
    command = [
        str(ENV_PYTHON), str(helper),
        "--repo-dir", str(REPO_DIR),
        "--drive-data-dir", str(DRIVE_DATA_DIR),
        "--local-data-dir", str(LOCAL_DATA_DIR),
        "--drive-output-dir", str(drive_output),
        "--local-output-dir", str(local_output),
        "--resume", RESUME_MODE,
        "--loss-mode", LOSS_MODE,
        "--max-length", str(max_length),
        "--max-epochs", str(MAX_EPOCHS),
        "--patience", str(PATIENCE),
        "--head-only-epochs", str(HEAD_ONLY_EPOCHS),
        "--unfreeze-top-layers", str(UNFREEZE_TOP_LAYERS),
        "--head-learning-rate", str(HEAD_LEARNING_RATE),
        "--encoder-learning-rate", str(ENCODER_LEARNING_RATE),
        "--weight-decay", str(WEIGHT_DECAY),
        "--dropout", str(DROPOUT),
        "--batch-size", str(PHYSICAL_BATCH_SIZE),
        "--gradient-accumulation", str(GRADIENT_ACCUMULATION),
        "--warmup-ratio", str(WARMUP_RATIO),
    ]
    if selection_only:
        command.append("--selection-only")
    print("Running:\n", " ".join(command))
    subprocess.run(command, check=True, env=run_env)

if RUN_MAX_LENGTH_128_CANDIDATE:
    candidate_drive = DRIVE_OUTPUT_DIR / "candidates"
    candidate_local = LOCAL_OUTPUT_DIR / "candidates"
    for max_length in (104, 128):
        run_candidate(max_length, candidate_drive / f"max_length_{max_length}", candidate_local / f"max_length_{max_length}", selection_only=True)
    scores = []
    for max_length in (104, 128):
        validation = json.loads((candidate_drive / f"max_length_{max_length}" / "metrics.json").read_text())["validation"]
        scores.append((float(validation["mcc"]), float(validation["auprc"]), max_length))
    selected_length = max(scores, key=lambda item: (item[0], item[1], -item[2]))[2]
    print("Validation-only candidates:", scores, "selected:", selected_length)
    run_candidate(selected_length, DRIVE_OUTPUT_DIR, LOCAL_OUTPUT_DIR, selection_only=False)
else:
    run_candidate(104, DRIVE_OUTPUT_DIR, LOCAL_OUTPUT_DIR, selection_only=False)

if LOCAL_HF_HOME.exists():
    shutil.copytree(LOCAL_HF_HOME, DRIVE_MODEL_CACHE, dirs_exist_ok=True)
    print("DNABERT2 cache synced to:", DRIVE_MODEL_CACHE)


## 7. Read metrics and plot the fair-comparison diagnostics


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, roc_curve

metrics = pd.read_csv(DRIVE_OUTPUT_DIR / "metrics.csv")
history = pd.read_csv(DRIVE_OUTPUT_DIR / "history.csv")
predictions = pd.read_csv(DRIVE_OUTPUT_DIR / "predictions.csv")
display(metrics)

test_metrics = metrics.loc[metrics["split"] == "test"].iloc[0]
print("Held-out test MCC:", round(float(test_metrics["mcc"]), 6))
print("Held-out test AUPRC:", round(float(test_metrics["auprc"]), 6))
print("Validation-selected threshold:", round(float(test_metrics["threshold"]), 6))

plot_dir = DRIVE_OUTPUT_DIR / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="train")
axes[0].plot(history["epoch"], history["validation_loss"], marker="o", label="validation")
axes[0].set(title="Loss", xlabel="epoch", ylabel="loss")
axes[0].legend()
axes[1].plot(history["epoch"], history["validation_mcc"], marker="o", label="MCC")
axes[1].plot(history["epoch"], history["validation_auprc"], marker="o", label="AUPRC")
axes[1].set(title="Validation selection metrics", xlabel="epoch")
axes[1].legend()
fig.tight_layout()
fig.savefig(plot_dir / "final_training_curves.png", dpi=160)
plt.show()

test = predictions[predictions["split"] == "test"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay.from_predictions(test["label"], test["prediction"], labels=[0, 1], display_labels=["non-promoter", "promoter"], colorbar=False, ax=axes[0])
axes[0].set_title("Held-out test confusion matrix")
fpr, tpr, _ = roc_curve(test["label"], test["probability"])
axes[1].plot(fpr, tpr, label=f"AUROC={float(test_metrics['auroc']):.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set(xlabel="false-positive rate", ylabel="true-positive rate")
axes[1].legend()
precision, recall, _ = precision_recall_curve(test["label"], test["probability"])
axes[2].plot(recall, precision, label=f"AUPRC={float(test_metrics['auprc']):.3f}")
axes[2].set(xlabel="recall", ylabel="precision")
axes[2].legend()
fig.tight_layout()
fig.savefig(plot_dir / "final_test_diagnostics.png", dpi=160)
plt.show()


## 8. Verify the saved artifacts


In [ ]:
required = [
    "config.json", "input_split_audit.json", "environment.json", "history.csv",
    "metrics.csv", "metrics.json", "predictions.csv", "manifest.json",
    "checkpoints/latest.pt", "checkpoints/best_validation_mcc.pt",
    "plots/final_training_curves.png", "plots/final_test_diagnostics.png",
]
missing = [name for name in required if not (DRIVE_OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError("Missing final DNABERT2 artifacts: " + ", ".join(missing))
print("Completed final artifact set:")
for name in required:
    print("-", DRIVE_OUTPUT_DIR / name)
